# Train a Linear Model

*Medium · from [mlcode](https://mlcode-io.vercel.app/problems/torch-train-linear)*

Write a real training loop in PyTorch.

Given `X` of shape `(n, d)` and `y` of shape `(n,)` as float tensors, build a
`torch.nn.Linear(d, 1)`, train it with plain SGD on mean squared error for
`epochs` full batch steps, and return the trained module.

The loop is four lines and always in this order:

```
optimizer.zero_grad()
loss = criterion(model(X).squeeze(-1), y)
loss.backward()
optimizer.step()
```

`zero_grad()` first is the one people forget. Gradients **accumulate** in
`.grad` by default, which is what makes gradient accumulation across
mini-batches possible, and what silently ruins a loop that does not clear
them: step two would descend the sum of two steps' gradients.

`model(X)` returns shape `(n, 1)` while `y` is `(n,)`. Left alone, broadcasting
turns the difference into an `(n, n)` matrix and the loss becomes meaningless
while still producing a number. Squeeze the output.

This one is scored on the result, not on exact values: your trained model has
to reach a mean squared error below `0.01` on the data it was given.

### Example

```
model = fit_linear(X, y, epochs=400, lr=0.05)
model(X).squeeze(-1)   # close to y
```

---

Write your solution in the cell below, then run the grading cell at the bottom. Your work is recorded against the same account as the site, so a pass here shows there.

Keep the solution in the function you were given. The grading cell saves your definitions and the lines around them back to the site, not the whole session, so anything you type to poke about is left behind rather than ending up in your saved answer.

In [ ]:
#@title Connect this notebook to mlcode { display-mode: "form" }
#@markdown Paste the code from the problem page, then run this cell.
CODE = ""  #@param {type:"string"}

SITE = "https://mlcode-io.vercel.app"
SLUG = "torch-train-linear"

import json, urllib.request, urllib.error

def _mlc_get(path):
    with urllib.request.urlopen(SITE + path, timeout=30) as r:
        return r.read()

# The harness is fetched rather than pasted in, so the cases run here exactly
# as they would on the site.
for _mlc_n in ("harness.py", "notebook.py"):
    with open("/content/" + _mlc_n, "wb") as _mlc_fh:
        _mlc_fh.write(_mlc_get("/" + _mlc_n))

try:
    _mlc_meta = json.loads(_mlc_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))
    print("connected:", _mlc_meta["title"])
    print("torch:", __import__("torch").__version__)
except urllib.error.HTTPError as e:
    print("could not connect:", json.loads(e.read()).get("error", e.reason))
    print("Get a fresh code from the problem page; they last three hours.")


In [ ]:
import torch


def fit_linear(X, y, epochs, lr):
    """Train a torch.nn.Linear(d, 1) with SGD on mean squared error.

    X:      torch.Tensor of shape (n, d), float32
    y:      torch.Tensor of shape (n,), float32
    epochs: int number of full batch steps
    lr:     float learning rate
    returns: the trained torch.nn.Module
    """
    pass

In [ ]:
#@title Grade this notebook { display-mode: "form" }
#@markdown Runs the hidden tests and records the attempt on the site.
import json, sys, time, urllib.request, urllib.error

sys.path.insert(0, "/content")
import harness

# The tests are fetched now rather than stored in the notebook, so a reader
# cannot read them before attempting the problem.
_mlc_tests = json.loads(_mlc_get(f"/api/colab?op=tests&code={CODE}&slug={SLUG}"))["tests"]

# `In` is the execution history, not the current cells: a wrong first attempt,
# a cell that errored and three re-runs are all still in it. Saving that would
# put something on the site that does not even replay. Rebuild the source from
# what is actually defined now instead, and keep the history only as a fallback
# for a solution written at module level.
def _mlc_collect():
    import ast, inspect, types

    # This notebook's own form cells are not part of anyone's answer.
    _cells = [c for c in In[1:] if not c.lstrip().startswith("#@title")]

    _imports, _seen = [], set()
    _consts = dict()
    for _cell in _cells:
        for _line in _cell.split("\n"):
            _t = _line.strip()
            if (_t.startswith("import ") or _t.startswith("from ")) and _t not in _seen:
                _seen.add(_t)
                _imports.append(_t)
        try:
            _tree = ast.parse(_cell)
        except SyntaxError:
            continue
        # Module level constants the functions below may close over. Last
        # assignment wins, because an edited cell is run again.
        for _node in _tree.body:
            if isinstance(_node, ast.Assign) and len(_node.targets) == 1:
                _target = _node.targets[0]
                if isinstance(_target, ast.Name):
                    _text = ast.get_source_segment(_cell, _node)
                    if _text and len(_text) <= 300:
                        # A plain literal can go above the definitions, where a
                        # default argument can still see it. Anything else may
                        # call one of them, so it has to come after.
                        try:
                            ast.literal_eval(_node.value)
                            _lit = True
                        except Exception:
                            _lit = False
                        _consts[_target.id] = (_text, _lit)

    _defs, _defined = [], set()
    for _name, _obj in list(globals().items()):
        if _name.startswith("_mlc_") or _name in ("In", "Out"):
            continue
        if isinstance(_obj, (types.FunctionType, type)) and getattr(_obj, "__module__", "") == "__main__":
            try:
                _lead = inspect.getcomments(_obj) or ""
                _defs.append(_lead + inspect.getsource(_obj).rstrip())
                _defined.add(_name)
            except (OSError, TypeError):
                pass

    # Nothing defined means the solution is not shaped like a function, so fall
    # back to what was typed rather than returning an empty file.
    if not _defs:
        return "\n\n".join(_cells)

    _early, _late = [], []
    for _name, _pair in _consts.items():
        if _name in _defined or _name.startswith("_mlc_") or _name in ("CODE", "SITE", "SLUG"):
            continue
        # What is kept is the line of source, not the value it produced, so how
        # large the value is does not matter. The length bound above is what
        # keeps a pasted-in data literal out.
        (_early if _pair[1] else _late).append(_pair[0])

    _blocks = ["\n".join(_imports), "\n".join(_early),
               "\n\n\n".join(_defs), "\n".join(_late)]
    return "\n\n\n".join(b for b in _blocks if b.strip()) + "\n"

_mlc_source = _mlc_collect()

_mlc_started = time.time()
_mlc_results = harness.run_tests(_mlc_tests, dict(globals()), 60, _mlc_source, "submit")
_mlc_elapsed = int((time.time() - _mlc_started) * 1000)

_mlc_passed = sum(1 for r in _mlc_results if r["passed"])
for _mlc_r in _mlc_results:
    print(("  pass  " if _mlc_r["passed"] else "  FAIL  ") + _mlc_r["name"])
    if not _mlc_r["passed"] and _mlc_r["message"]:
        print("        " + _mlc_r["message"].replace("\n", "\n        "))
print(f"\n{_mlc_passed}/{len(_mlc_results)} passed")

_mlc_body = json.dumps({
    "code": CODE, "slug": SLUG, "passedCount": _mlc_passed,
    "totalCount": len(_mlc_results), "runtimeMs": _mlc_elapsed, "source": _mlc_source,
}).encode()
_mlc_req = urllib.request.Request(SITE + "/api/colab", data=_mlc_body,
                              headers={"Content-Type": "application/json"})
try:
    with urllib.request.urlopen(_mlc_req, timeout=30) as r:
        print("recorded on mlcode:", "solved" if json.load(r)["passed"] else "attempt saved")
except urllib.error.HTTPError as e:
    print("could not record:", json.loads(e.read()).get("error", e.reason))
